In [ ]:
"""
download_netcdf_data.py
=======================
Módulo de aquisição de dados NetCDF4 do satélite GOES-16 a partir do bucket
público da NOAA na AWS S3.

Fluxo principal:
    1. `available_products`  – consulta/exibe produtos disponíveis.
    2. `get_netcdf_data`     – lista arquivos no S3 para um intervalo de datas.
    3. `filter_files`        – seleciona apenas as bandas espectrais de interesse.
    4. `download_files`      – faz o download dos arquivos filtrados para disco.

Dependências:
    - s3fs

Uso típico:
    product = available_products('codes')[9]          # ex.: 'ABI-L2-ACMF'
    files, errors = get_netcdf_data(product, 2020, 2020, 151, 213, '16')
    filtered = filter_files(files)
    download_files(product, filtered, local_base_dir='Dados')
"""

import s3fs

# ---------------------------------------------------------------------------
# Constantes do módulo
# ---------------------------------------------------------------------------

# Bucket público da NOAA para dados GOES-16
S3_BUCKET = "noaa-goes16"

# Bandas espectrais utilizadas no cálculo dos índices (M6C01–M6C06 exceto M6C04)
# C01 – Azul, C02 – Vermelho, C03 – NIR, C05 – SWIR1, C06 – SWIR2
BANDS_OF_INTEREST = ["M6C01", "M6C02", "M6C03", "M6C05", "M6C06"]

# Catálogo de produtos GOES-16 disponíveis no S3
GOES16_PRODUCTS = [
    ("ABI-L1b-RadF",   "Advanced Baseline Imager Level 1b Full Disk"),
    ("ABI-L1b-RadC",   "Advanced Baseline Imager Level 1b CONUS"),
    ("ABI-L1b-RadM",   "Advanced Baseline Imager Level 1b Mesoscale"),
    ("ABI-L2-ACHAC",   "ABI Level 2 Cloud Top Height CONUS"),
    ("ABI-L2-ACHAF",   "ABI Level 2 Cloud Top Height Full Disk"),
    ("ABI-L2-ACHAM",   "ABI Level 2 Cloud Top Height Mesoscale"),
    ("ABI-L2-ACHTF",   "ABI Level 2 Cloud Top Temperature Full Disk"),
    ("ABI-L2-ACHTM",   "ABI Level 2 Cloud Top Temperature Mesoscale"),
    ("ABI-L2-ACMC",    "ABI Level 2 Clear Sky Mask CONUS"),
    ("ABI-L2-ACMF",    "ABI Level 2 Clear Sky Mask Full Disk"),
    ("ABI-L2-ACMM",    "ABI Level 2 Clear Sky Mask Mesoscale"),
    ("ABI-L2-ACTPC",   "ABI Level 2 Cloud Top Phase CONUS"),
    ("ABI-L2-ACTPF",   "ABI Level 2 Cloud Top Phase Full Disk"),
    ("ABI-L2-ACTPM",   "ABI Level 2 Cloud Top Phase Mesoscale"),
    ("ABI-L2-ADPC",    "ABI Level 2 Aerosol Detection CONUS"),
    ("ABI-L2-ADPF",    "ABI Level 2 Aerosol Detection Full Disk"),
    ("ABI-L2-ADPM",    "ABI Level 2 Aerosol Detection Mesoscale"),
    ("ABI-L2-AODC",    "ABI Level 2 Aerosol Optical Depth CONUS"),
    ("ABI-L2-AODF",    "ABI Level 2 Aerosol Optical Depth Full Disk"),
    ("ABI-L2-CMIPC",   "ABI Level 2 Cloud and Moisture Imagery CONUS"),
    ("ABI-L2-CMIPF",   "ABI Level 2 Cloud and Moisture Imagery Full Disk"),
    ("ABI-L2-CMIPM",   "ABI Level 2 Cloud and Moisture Imagery Mesoscale"),
    ("ABI-L2-CODC",    "ABI Level 2 Cloud Optical Depth CONUS"),
    ("ABI-L2-CODF",    "ABI Level 2 Cloud Optical Depth Full Disk"),
    ("ABI-L2-CPSC",    "ABI Level 2 Cloud Particle Size CONUS"),
    ("ABI-L2-CPSF",    "ABI Level 2 Cloud Particle Size Full Disk"),
    ("ABI-L2-CPSM",    "ABI Level 2 Cloud Particle Size Mesoscale"),
    ("ABI-L2-CTPC",    "ABI Level 2 Cloud Top Pressure CONUS"),
    ("ABI-L2-CTPF",    "ABI Level 2 Cloud Top Pressure Full Disk"),
    ("ABI-L2-DMWC",    "ABI Level 2 Derived Motion Winds CONUS"),
    ("ABI-L2-DMWF",    "ABI Level 2 Derived Motion Winds Full Disk"),
    ("ABI-L2-DMWM",    "ABI Level 2 Derived Motion Winds Mesoscale"),
    ("ABI-L2-DSIC",    "ABI Level 2 Derived Stability Indices CONUS"),
    ("ABI-L2-DSIF",    "ABI Level 2 Derived Stability Indices Full Disk"),
    ("ABI-L2-DSIM",    "ABI Level 2 Derived Stability Indices Mesoscale"),
    ("ABI-L2-DSRC",    "ABI Level 2 Downward Shortwave Radiation CONUS"),
    ("ABI-L2-DSRF",    "ABI Level 2 Downward Shortwave Radiation Full Disk"),
    ("ABI-L2-DSRM",    "ABI Level 2 Downward Shortwave Radiation Mesoscale"),
    ("ABI-L2-FDCC",    "ABI Level 2 Fire (Hot Spot Characterization) CONUS"),
    ("ABI-L2-FDCF",    "ABI Level 2 Fire (Hot Spot Characterization) Full Disk"),
    ("ABI-L2-LSTC",    "ABI Level 2 Land Surface Temperature CONUS"),
    ("ABI-L2-LSTF",    "ABI Level 2 Land Surface Temperature Full Disk"),
    ("ABI-L2-LSTM",    "ABI Level 2 Land Surface Temperature Mesoscale"),
    ("ABI-L2-LVMPC",   "ABI Level 2 Legacy Vertical Moisture Profile CONUS"),
    ("ABI-L2-LVMPF",   "ABI Level 2 Legacy Vertical Moisture Profile Full Disk"),
    ("ABI-L2-LVMPM",   "ABI Level 2 Legacy Vertical Moisture Profile Mesoscale"),
    ("ABI-L2-LVTPC",   "ABI Level 2 Legacy Vertical Temperature Profile CONUS"),
    ("ABI-L2-LVTPF",   "ABI Level 2 Legacy Vertical Temperature Profile Full Disk"),
    ("ABI-L2-LVTPM",   "ABI Level 2 Legacy Vertical Temperature Profile Mesoscale"),
    ("ABI-L2-MCMIPC",  "ABI Level 2 Multi-Band Cloud and Moisture Imagery CONUS"),
    ("ABI-L2-MCMIPF",  "ABI Level 2 Multi-Band Cloud and Moisture Imagery Full Disk"),
    ("ABI-L2-MCMIPM",  "ABI Level 2 Multi-Band Cloud and Moisture Imagery Mesoscale"),
    ("ABI-L2-RRQPEF",  "ABI Level 2 Rainfall Rate (QPE) Full Disk"),
    ("ABI-L2-RSRC",    "ABI Level 2 Reflected Shortwave Radiation TOA CONUS"),
    ("ABI-L2-RSRF",    "ABI Level 2 Reflected Shortwave Radiation TOA Full Disk"),
    ("ABI-L2-SSTF",    "ABI Level 2 Sea Surface (Skin) Temperature Full Disk"),
    ("ABI-L2-TPWC",    "ABI Level 2 Total Precipitable Water CONUS"),
    ("ABI-L2-TPWF",    "ABI Level 2 Total Precipitable Water Full Disk"),
    ("ABI-L2-TPWM",    "ABI Level 2 Total Precipitable Water Mesoscale"),
    ("ABI-L2-VAAF",    "ABI Level 2 Volcanic Ash Detection Full Disk"),
    ("GLM-L2-LCFA",    "Geostationary Lightning Mapper Level 2 Lightning Detection"),
    ("SUVI-L1b-Fe093", "Solar Ultraviolet Imager Level 1b Extreme Ultraviolet (Fe 9.3 nm)"),
    ("SUVI-L1b-Fe131", "Solar Ultraviolet Imager Level 1b Extreme Ultraviolet (Fe 13.1 nm)"),
    ("SUVI-L1b-Fe171", "Solar Ultraviolet Imager Level 1b Extreme Ultraviolet (Fe 17.1 nm)"),
    ("SUVI-L1b-Fe195", "Solar Ultraviolet Imager Level 1b Extreme Ultraviolet (Fe 19.5 nm)"),
    ("SUVI-L1b-Fe284", "Solar Ultraviolet Imager Level 1b Extreme Ultraviolet (Fe 28.4 nm)"),
    ("SUVI-L1b-He303", "Solar Ultraviolet Imager Level 1b Extreme Ultraviolet (He 30.3 nm)"),
]


# ---------------------------------------------------------------------------
# Funções públicas
# ---------------------------------------------------------------------------

def available_products(operation: str):
    """
    Consulta ou exibe os produtos GOES-16 disponíveis no S3 da NOAA.

    Parâmetros:
        operation (str): Tipo de saída desejada.
            - 'codes'       : Retorna lista com os códigos dos produtos.
            - 'description' : Imprime tabela com índice, código e descrição.

    Retorna:
        list[str] | None:
            Lista de códigos se operation='codes'; None se operation='description'.

    Exemplos:
        >>> codes = available_products('codes')
        >>> available_products('description')
    """
    if operation == 'codes':
        return [code for code, _ in GOES16_PRODUCTS]

    if operation == 'description':
        header = f"{'#':<5} | {'Produto':<25} | Descrição"
        print(header)
        print("-" * 95)
        for i, (code, description) in enumerate(GOES16_PRODUCTS):
            print(f"{i:<5} | {code:<25} | {description}")
        return None

    raise ValueError(f"Operação inválida: '{operation}'. Use 'codes' ou 'description'.")


def get_netcdf_data(
    product: str,
    year_start: int,
    year_end: int,
    day_start: int,
    day_end: int,
    hour: str,
) -> tuple[list[list[str]], list[tuple[int, int]]]:
    """
    Lista arquivos NetCDF4 do GOES-16 no S3 para um intervalo de anos, dias e hora.

    O acesso é feito de forma anônima ao bucket público 'noaa-goes16' da NOAA.
    Combinações (ano, dia) onde o caminho não existe ou houve erro são registradas
    separadamente para facilitar diagnóstico.

    Parâmetros:
        product   (str): Código do produto (ex.: 'ABI-L2-CMIPF').
        year_start (int): Ano inicial do intervalo (inclusive).
        year_end   (int): Ano final do intervalo (inclusive).
        day_start  (int): Dia juliano inicial (inclusive), entre 1 e 366.
        day_end    (int): Dia juliano final (inclusive), entre 1 e 366.
        hour       (str): Hora UTC no formato 'HH' (ex.: '16').

    Retorna:
        tuple:
            files  (list[list[str]]): Cada elemento é a lista de arquivos de um
                                      (ano, dia) bem-sucedido.
            errors (list[tuple[int, int]]): Pares (ano, dia) com falha de acesso.

    Exemplos:
        >>> files, errors = get_netcdf_data('ABI-L2-CMIPF', 2020, 2020, 151, 213, '16')
        >>> print(f"{len(files)} dias encontrados, {len(errors)} com erro.")
    """
    # Valida o formato da hora antes de iniciar as requisições
    if not (hour.isdigit() and 0 <= int(hour) <= 23):
        raise ValueError(f"Hora inválida: '{hour}'. Use formato 'HH' entre '00' e '23'.")

    fs = s3fs.S3FileSystem(anon=True)
    files = []
    errors = []

    for year in range(year_start, year_end + 1):
        for day in range(day_start, day_end + 1):
            # Caminho no S3: noaa-goes16/<produto>/<ano>/<dia 3 dígitos>/<hora>/
            s3_path = f"{S3_BUCKET}/{product}/{year}/{day:03d}/{hour}/"
            try:
                day_files = fs.ls(s3_path)
                files.append(day_files)
            except Exception:
                # Registra o par (ano, dia) para diagnóstico posterior
                errors.append((year, day))

    return files, errors


def _has_bands(file_list: list[list[str]]) -> bool:
    """
    Verifica se algum arquivo na listagem contém bandas do tipo 'M6C##'.

    Usada internamente por `filter_files` para decidir a estratégia de filtragem.

    Parâmetros:
        file_list (list[list[str]]): Listagem retornada por `get_netcdf_data`.

    Retorna:
        bool: True se ao menos um arquivo contiver 'M6C' no nome.
    """
    return any(
        "M6C" in filename
        for daily_files in file_list
        for filename in daily_files
    )


def filter_files(file_list: list[list[str]]) -> list[str]:
    """
    Filtra os arquivos listados pelo S3, retornando uma lista plana de caminhos.

    Estratégia de filtragem:
        - Se os arquivos contiverem bandas (padrão 'M6C##'): retorna apenas as
          bandas definidas em BANDS_OF_INTEREST (C01, C02, C03, C05, C06),
          correspondendo a azul, vermelho, NIR, SWIR1 e SWIR2.
        - Caso contrário (ex.: produtos L2 sem bandas): retorna todos os arquivos.

    Parâmetros:
        file_list (list[list[str]]): Listagem retornada por `get_netcdf_data`,
                                     onde cada elemento é a lista de arquivos de
                                     um (ano, dia).

    Retorna:
        list[str]: Lista plana com os caminhos dos arquivos selecionados.

    Exemplos:
        >>> filtered = filter_files(files)
        >>> print(f"{len(filtered)} arquivos selecionados.")
    """
    filtered = []

    if _has_bands(file_list):
        # Mantém apenas as bandas de interesse para cálculo dos índices espectrais
        for daily_files in file_list:
            for band in BANDS_OF_INTEREST:
                matched = [f for f in daily_files if band in f]
                filtered.extend(matched)
    else:
        # Produtos sem bandas (ex.: ACMF, FDCF): retorna todos os arquivos
        for daily_files in file_list:
            filtered.extend(daily_files)

    return filtered


def download_files(
    product: str,
    file_list: list[str],
    local_base_dir: str,
) -> list[str]:
    """
    Faz o download de arquivos NetCDF4 do S3 para o diretório local.

    Os arquivos são salvos em:
        <local_base_dir>/<product>/netCDF/<nome_do_arquivo>

    Parâmetros:
        product       (str): Código do produto (ex.: 'ABI-L2-CMIPF').
        file_list     (list[str]): Lista de caminhos S3 retornada por `filter_files`.
        local_base_dir (str): Diretório base local onde os arquivos serão salvos.

    Retorna:
        list[str]: Lista de caminhos locais dos arquivos baixados com sucesso.

    Exemplos:
        >>> downloaded = download_files('ABI-L2-CMIPF', filtered, 'Dados')
        >>> print(f"{len(downloaded)} arquivos baixados.")
    """
    import os

    fs = s3fs.S3FileSystem(anon=True)

    # Garante que o diretório de destino existe
    local_dir = os.path.join(local_base_dir, product, "netCDF")
    os.makedirs(local_dir, exist_ok=True)

    downloaded = []
    total = len(file_list)

    for idx, remote_path in enumerate(file_list, start=1):
        filename = remote_path.split("/")[-1]
        local_path = os.path.join(local_dir, filename)

        try:
            fs.get(remote_path, local_path)
            print(f"[{idx}/{total}] Download concluído: {filename}")
            downloaded.append(local_path)
        except Exception as e:
            print(f"[{idx}/{total}] Erro ao baixar '{filename}': {e}")

    print(f"\nDownload finalizado: {len(downloaded)}/{total} arquivos obtidos com sucesso.")
    return downloaded


# ---------------------------------------------------------------------------
# Exemplo de uso (executado apenas quando o script é chamado diretamente)
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Índices do catálogo: 9 = ABI-L2-ACMF | 20 = ABI-L2-CMIPF | 39 = ABI-L2-FDCF
    PRODUCT = available_products('codes')[20]
    LOCAL_DIR = "Dados"

    print(f"Produto selecionado: {PRODUCT}\n")

    files, errors = get_netcdf_data(
        product=PRODUCT,
        year_start=2020,
        year_end=2020,
        day_start=130,
        day_end=240,
        hour="13",
    )

    print(f"Dias encontrados : {len(files)}")
    print(f"Dias com erro    : {len(errors)}")
    if errors:
        print(f"  → Erros: {errors}")

    filtered = filter_files(files)
    print(f"Arquivos filtrados: {len(filtered)}\n")

    downloaded = download_files(PRODUCT, filtered, LOCAL_DIR)

Produto selecionado: ABI-L2-CMIPF

Dias encontrados : 111
Dias com erro    : 0
Arquivos filtrados: 3330

[1/3330] Download concluído: OR_ABI-L2-CMIPF-M6C01_G16_s20201301300147_e20201301309455_c20201301309524.nc
[2/3330] Download concluído: OR_ABI-L2-CMIPF-M6C01_G16_s20201301310147_e20201301319455_c20201301319525.nc
[3/3330] Download concluído: OR_ABI-L2-CMIPF-M6C01_G16_s20201301320147_e20201301329455_c20201301329521.nc
[4/3330] Download concluído: OR_ABI-L2-CMIPF-M6C01_G16_s20201301330147_e20201301339455_c20201301339528.nc
[5/3330] Download concluído: OR_ABI-L2-CMIPF-M6C01_G16_s20201301340147_e20201301349455_c20201301349547.nc
[6/3330] Download concluído: OR_ABI-L2-CMIPF-M6C01_G16_s20201301350147_e20201301359455_c20201301359537.nc
